In [1]:
# Enable inline plotting for Jupyter notebooks
%matplotlib inline

# Import required libraries for visualization
import warnings
warnings.filterwarnings('ignore')

print("Notebook configured for interactive visualization")

Notebook configured for interactive visualization


In [2]:
conda install -c conda-forge jupyterlab_widgets notebook

Retrieving notices: / Retrying (Retry(total=2, connect=None, read=None, redirect=None, status=None)) after connection broken by 'NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x73152822b6b0>: Failed to resolve 'repo.anaconda.com' ([Errno -3] Temporary failure in name resolution)")': /pkgs/main/notices.json

Retrying (Retry(total=2, connect=None, read=None, redirect=None, status=None)) after connection broken by 'NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x73152822bcb0>: Failed to resolve 'conda.anaconda.org' ([Errno -3] Temporary failure in name resolution)")': /conda-forge/notices.json

- Retrying (Retry(total=2, connect=None, read=None, redirect=None, status=None)) after connection broken by 'NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x73152822b860>: Failed to resolve 'repo.anaconda.com' ([Errno -3] Temporary failure in name resolution)")': /pkgs/r/notices.json

Retrying (Retry(total=2, connect=None, read=None, re

In [3]:
!pip install matplotlib numpy SimpleITK opencv-python


In [4]:
# Install required packages
!pip install antspyx SimpleITK matplotlib

In [5]:
!pip install torchio

In [6]:
conda install -c conda-forge opencv

Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: done
Solving environment: done
done

# All requested packages already installed.


# All requested packages already installed.


Note: you may need to restart the kernel to use updated packages.

Note: you may need to restart the kernel to use updated packages.


In [7]:
conda install -c conda-forge ipywidgets=7.7.1


Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: done
Solving environment/ done
done

# All requested packages already installed.


# All requested packages already installed.


Note: you may need to restart the kernel to use updated packages.

Note: you may need to restart the kernel to use updated packages.


Helper functions

In [8]:
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, fixed, widgets
from IPython.display import display, clear_output
import numpy as np
import SimpleITK as sitk
import cv2

def explore_3D_array(arr: np.ndarray, cmap: str = 'gray'):
  """
  Given a 3D array with shape (Z,X,Y) This function will create an interactive
  widget to check out all the 2D arrays with shape (X,Y) inside the 3D array. 
  The purpose of this function to visual inspect the 2D arrays in the image. 

  Args:
    arr : 3D array with shape (Z,X,Y) that represents the volume of a MRI image
    cmap : Which color map use to plot the slices in matplotlib.pyplot
  """
  # Create the output widget
  output = widgets.Output()
  
  def view_slice(SLICE):
    with output:
      clear_output(wait=True)
      plt.figure(figsize=(7,7))
      plt.imshow(arr[SLICE, :, :], cmap=cmap)
      plt.title(f'Slice {SLICE}')
      plt.colorbar()
      plt.tight_layout()
  
  # Create the widget
  slider = widgets.IntSlider(min=0, max=arr.shape[0]-1, step=1, description='Slice:')
  
  # Link the widget to the function
  ui = interactive(view_slice, SLICE=slider)
  
  # Display the widget and output area
  display(ui, output)
  
  # Show initial slice
  view_slice(0)


def explore_3D_array_comparison(arr_before: np.ndarray, arr_after: np.ndarray, cmap: str = 'gray'):
  """
  Given two 3D arrays with shape (Z,X,Y) This function will create an interactive
  widget to check out all the 2D arrays with shape (X,Y) inside the 3D arrays.
  The purpose of this function to visual compare the 2D arrays after some transformation. 

  Args:
    arr_before : 3D array with shape (Z,X,Y) that represents the volume of a MRI image, before any transform
    arr_after : 3D array with shape (Z,X,Y) that represents the volume of a MRI image, after some transform    
    cmap : Which color map use to plot the slices in matplotlib.pyplot
  """
  assert arr_after.shape == arr_before.shape
  
  # Create the output widget
  output = widgets.Output()
  
  def view_comparison(SLICE):
    with output:
      clear_output(wait=True)
      fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
      
      # Left image (Before)
      ax1.imshow(arr_before[SLICE, :, :], cmap=cmap)
      ax1.axis('on')
      
      # Right image (After)
      ax2.imshow(arr_after[SLICE, :, :], cmap=cmap)
      ax2.axis('on')
      
      plt.tight_layout()
      plt.show()
  
  # Create the widget
  slider = widgets.IntSlider(min=0, max=arr_before.shape[0]-1, step=1, description='Slice:')
  
  # Link the widget to the function
  ui = interactive(view_comparison, SLICE=slider)
  
  # Display the widget and output area
  display(ui, output)
  
  # Show initial slice
  view_comparison(0)


def show_sitk_img_info(img: sitk.Image):
  """
  Given a sitk.Image instance prints the information about the MRI image contained.

  Args:
    img : instance of the sitk.Image to check out
  """
  pixel_type = img.GetPixelIDTypeAsString()
  origin = img.GetOrigin()
  dimensions = img.GetSize()
  spacing = img.GetSpacing()
  direction = img.GetDirection()

  info = {'Pixel Type' : pixel_type, 'Dimensions': dimensions, 'Spacing': spacing, 'Origin': origin,  'Direction' : direction}
  for k,v in info.items():
    print(f' {k} : {v}')


def add_suffix_to_filename(filename: str, suffix:str) -> str:
  """
  Takes a NIfTI filename and appends a suffix.

  Args:
      filename : NIfTI filename
      suffix : suffix to append

  Returns:
      str : filename after append the suffix
  """
  if filename.endswith('.nii'):
      result = filename.replace('.nii', f'_{suffix}.nii')
      return result
  elif filename.endswith('.nii.gz'):
      result = filename.replace('.nii.gz', f'_{suffix}.nii.gz')
      return result
  else:
      raise RuntimeError('filename with unknown extension')


def rescale_linear(array: np.ndarray, new_min: int, new_max: int):
  """Rescale an array linearly."""
  minimum, maximum = np.min(array), np.max(array)
  m = (new_max - new_min) / (maximum - minimum)
  b = new_min - m * minimum
  return m * array + b


def explore_3D_array_with_mask_contour(arr: np.ndarray, mask: np.ndarray, thickness: int = 1):
  """
  Given a 3D array with shape (Z,X,Y) This function will create an interactive
  widget to check out all the 2D arrays with shape (X,Y) inside the 3D array. The binary
  mask provided will be used to overlay contours of the region of interest over the 
  array. The purpose of this function is to visual inspect the region delimited by the mask.

  Args:
    arr : 3D array with shape (Z,X,Y) that represents the volume of a MRI image
    mask : binary mask to obtain the region of interest
  """
  assert arr.shape == mask.shape
  
  _arr = rescale_linear(arr,0,1)
  _mask = rescale_linear(mask,0,1)
  _mask = _mask.astype(np.uint8)
  
  # Create the output widget
  output = widgets.Output()
  
  def view_contour(SLICE):
    with output:
      clear_output(wait=True)
      plt.figure(figsize=(7,7))
      
      arr_rgb = cv2.cvtColor(_arr[SLICE, :, :], cv2.COLOR_GRAY2RGB)
      contours, _ = cv2.findContours(_mask[SLICE, :, :], cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
      
      arr_with_contours = cv2.drawContours(arr_rgb, contours, -1, (0,1,0), thickness)

      plt.imshow(arr_with_contours)
      plt.title(f'Slice {SLICE} with contours')
      plt.tight_layout()
      plt.show()
  
  # Create the widget
  slider = widgets.IntSlider(min=0, max=arr.shape[0]-1, step=1, description='Slice:')
  
  # Link the widget to the function
  ui = interactive(view_contour, SLICE=slider)
  
  # Display the widget and output area
  display(ui, output)
  
  # Show initial slice
  view_contour(0)

Data Loader


In [9]:
import os
import glob
import nibabel as nib
import numpy as np

# --- Configuration ---
BASE_DIR = '/mnt/Data/AKIB/Training data'
TARGET_SEQUENCE = 'T1' # Focusing on T1 for a start, as it is standard. You can loop through all three later.
NUM_SUBJECTS = 50

def load_image(filepath):
    """Loads a NIfTI file and returns the data array and NIfTI object."""
    try:
        img = nib.load(filepath)
        # Using .get_fdata() to get the image data as a floating-point array
        data = img.get_fdata()
        return data, img
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return None, None

def organize_data_paths(base_dir, num_subjects, sequence):
    """Generates a list of dictionaries with paired file paths for all subjects."""
    data_list = []
    
    # Subject folders are named 1, 2, 3, ...
    for sub_id in range(1, num_subjects + 1):
        subject_dir = os.path.join(base_dir, str(sub_id))
        
        # Build file paths
        ulf_path = os.path.join(subject_dir, '64mT', f'{sub_id}_{sequence}.nii.gz')
        hf_path = os.path.join(subject_dir, '3T', f'{sub_id}_{sequence}.nii.gz')
        
        # Check if files exist before adding to the list
        if os.path.exists(ulf_path) and os.path.exists(hf_path):
            data_list.append({
                'subject_id': sub_id,
                'ulf_path': ulf_path,
                'hf_path': hf_path
            })
        else:
            print(f"Warning: Missing files for Subject {sub_id} ({sequence}). Skipping.")
            
    return data_list

# Get the list of file pairs
data_pairs = organize_data_paths(BASE_DIR, NUM_SUBJECTS, TARGET_SEQUENCE)
print(f"Found {len(data_pairs)} valid T1-weighted subject pairs.")
data_pairs[0]['ulf_path']

Found 50 valid T1-weighted subject pairs.


'/mnt/Data/AKIB/Training data/1/64mT/1_T1.nii.gz'

In [10]:
import os
from helpers import *

import ants
import SimpleITK as sitk

print(f'AntsPy version = {ants.__version__}')
print(f'SimpleITK version = {sitk.__version__}')

AntsPy version = 0.6.1
SimpleITK version = 2.5.2


In [11]:
# Visualize the original 3D images before registration and preprocessing
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt

# Get the sample pair that we'll process
sample_pair = data_pairs[0]
print(f"Visualizing Subject {sample_pair['subject_id']} before registration...")

# Load the original images
ulf_sitk_orig = sitk.ReadImage(sample_pair['ulf_path'], sitk.sitkFloat32)
hf_sitk_orig = sitk.ReadImage(sample_pair['hf_path'], sitk.sitkFloat32)

# Convert to numpy arrays for visualization
ulf_orig = sitk.GetArrayFromImage(ulf_sitk_orig)
hf_orig = sitk.GetArrayFromImage(hf_sitk_orig)

print(f"Original ULF shape: {ulf_orig.shape}")
print(f"Original HF shape: {hf_orig.shape}")
print(f"ULF spacing: {ulf_sitk_orig.GetSpacing()}")
print(f"HF spacing: {hf_sitk_orig.GetSpacing()}")

# Visualize using the interactive slider function
print("\nInteractive visualization of ULF and HF images (before registration):")
# Show the ULF and HF images side by side with slider
if ulf_orig.shape[0] == hf_orig.shape[0]:
    explore_3D_array_comparison(ulf_orig, hf_orig, cmap='gray')
else:
    print("\nULF and HF have different numbers of slices, cannot directly compare side by side.")
    print(f"ULF has {ulf_orig.shape[0]} slices, HF has {hf_orig.shape[0]} slices.")

Visualizing Subject 1 before registration...
Original ULF shape: (160, 224, 224)
Original HF shape: (160, 224, 224)
ULF spacing: (1.0, 1.0, 1.0)
HF spacing: (1.0, 1.0, 1.0)

Interactive visualization of ULF and HF images (before registration):


interactive(children=(IntSlider(value=79, description='SLICE', max=159), Output()), _dom_classes=('widget-inte…

Image registration

In [12]:
import SimpleITK as sitk

def preprocess_volumes(ulf_path, hf_path, target_spacing=(1.0, 1.0, 1.0)):
    # Load with SimpleITK
    ulf_sitk = sitk.ReadImage(ulf_path, sitk.sitkFloat32)
    hf_sitk = sitk.ReadImage(hf_path, sitk.sitkFloat32)

    # --- 1. Registration (Aligning ULF to HF) ---
    # Use a simple rigid registration to fine-tune alignment
    R = sitk.ImageRegistrationMethod()
    R.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    R.SetOptimizerAsRegularStepGradientDescent(learningRate=2.0, numberOfIterations=100, minStep=0.01)
    R.SetInterpolator(sitk.sitkLinear)
    
    # Use the HF image as the fixed image and the ULF image as the moving image
    #### Place moving image to fixed image space
    
    initial_transform = sitk.CenteredTransformInitializer(
        hf_sitk, ulf_sitk, sitk.Euler3DTransform(), sitk.CenteredTransformInitializerFilter.GEOMETRY
    )
    
    # Set the initial transform before executing
    R.SetInitialTransform(initial_transform)
    
    # Execute registration with just the fixed and moving images
    final_transform = R.Execute(hf_sitk, ulf_sitk)
    
    # Apply the final transform to the ULF image
    resampled_ulf = sitk.Resample(
        ulf_sitk, hf_sitk, final_transform, sitk.sitkBSpline, 0.0, ulf_sitk.GetPixelID()
    )
    
    # --- 2. Resampling (Setting Isotropic Resolution) ---
    # Define a common reference space for isotropic 1mm resolution
    
    # Calculate new size to maintain physical size when changing spacing
    new_size = [int(sz * orig_sp / target_sp) for sz, orig_sp, target_sp in 
               zip(hf_sitk.GetSize(), hf_sitk.GetSpacing(), target_spacing)]
    
    # Create a reference image with the desired spacing
    reference_image = sitk.Image(new_size, hf_sitk.GetPixelID())
    reference_image.SetOrigin(hf_sitk.GetOrigin())
    reference_image.SetSpacing(target_spacing)
    reference_image.SetDirection(hf_sitk.GetDirection())
    
    # Resample both images to the new isotropic reference space
    resampled_hf = sitk.Resample(hf_sitk, reference_image)
    resampled_ulf = sitk.Resample(resampled_ulf, reference_image) # Use the aligned ULF

    # Convert back to numpy array for deep learning
    ulf_data_out = sitk.GetArrayFromImage(resampled_ulf)
    hf_data_out = sitk.GetArrayFromImage(resampled_hf)
    
    return ulf_data_out, hf_data_out

# --- Example Usage (Process one subject) ---
sample_pair = data_pairs[0]
print(f"\nProcessing Subject {sample_pair['subject_id']}...")

ulf_processed, hf_processed = preprocess_volumes(sample_pair['ulf_path'], sample_pair['hf_path'])
print(f"Processed ULF shape: {ulf_processed.shape}")
print(f"Processed HF shape: {hf_processed.shape}")

# Now, loop through all data pairs, save the processed NumPy files, 
# and move on to normalization and patching (next steps).


Processing Subject 1...
Processed ULF shape: (160, 224, 224)
Processed HF shape: (160, 224, 224)
Processed ULF shape: (160, 224, 224)
Processed HF shape: (160, 224, 224)


In [13]:
# Visualize the processed and registered images
print("Visualizing the processed images after registration")
print("ULF shape:", ulf_processed.shape)
print("HF shape:", hf_processed.shape)

ulf_unprocessed = sitk.GetArrayFromImage(sitk.ReadImage(sample_pair['ulf_path'], sitk.sitkFloat32))
hf_unprocessed = sitk.GetArrayFromImage(sitk.ReadImage(sample_pair['hf_path'], sitk.sitkFloat32))

# Use our comparison function to view both images side by side
explore_3D_array_comparison(ulf_unprocessed, ulf_processed, cmap='gray')
print("Interactive visualization launched. Use the slider to browse through slices.")

Visualizing the processed images after registration
ULF shape: (160, 224, 224)
HF shape: (160, 224, 224)


interactive(children=(IntSlider(value=79, description='SLICE', max=159), Output()), _dom_classes=('widget-inte…

Interactive visualization launched. Use the slider to browse through slices.


In [14]:
# Use our comparison function to view both images side by side
explore_3D_array_comparison(hf_unprocessed, hf_processed, cmap='gray')
print("Interactive visualization launched. Use the slider to browse through slices.")

interactive(children=(IntSlider(value=79, description='SLICE', max=159), Output()), _dom_classes=('widget-inte…

Interactive visualization launched. Use the slider to browse through slices.


In [15]:
# Process and save multiple subjects
import os

def process_and_save_all_subjects(data_pairs, output_dir='/mnt/Data/AKIB/processed_data', max_subjects=50):
    """
    Process multiple subjects and save the registered and resampled data
    
    Args:
        data_pairs: List of dictionaries containing file paths for each subject
        output_dir: Directory to save processed data
        max_subjects: Maximum number of subjects to process (for testing)
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Process each subject
    for i, subject_data in enumerate(data_pairs[:max_subjects]):
        subject_id = subject_data['subject_id']
        print(f"Processing subject {subject_id} ({i+1}/{min(len(data_pairs), max_subjects)})...")
        
        try:
            # Process the volumes
            ulf_processed, hf_processed = preprocess_volumes(
                subject_data['ulf_path'], 
                subject_data['hf_path']
            )
            
            # Save as numpy arrays
            subject_dir = os.path.join(output_dir, f"subject_{subject_id}")
            os.makedirs(subject_dir, exist_ok=True)
            
            np.save(os.path.join(subject_dir, 'ulf_processed.npy'), ulf_processed)
            np.save(os.path.join(subject_dir, 'hf_processed.npy'), hf_processed)
            
            print(f"Subject {subject_id} processed and saved successfully!")
            
        except Exception as e:
            print(f"Error processing subject {subject_id}: {str(e)}")
    
    print("Processing complete!")

process_and_save_all_subjects(data_pairs, max_subjects=50)

Processing subject 1 (1/50)...
Subject 1 processed and saved successfully!
Processing subject 2 (2/50)...
Subject 1 processed and saved successfully!
Processing subject 2 (2/50)...
Subject 2 processed and saved successfully!
Processing subject 3 (3/50)...
Subject 2 processed and saved successfully!
Processing subject 3 (3/50)...
Subject 3 processed and saved successfully!
Processing subject 4 (4/50)...
Subject 3 processed and saved successfully!
Processing subject 4 (4/50)...
Subject 4 processed and saved successfully!
Processing subject 5 (5/50)...
Subject 4 processed and saved successfully!
Processing subject 5 (5/50)...
Subject 5 processed and saved successfully!
Processing subject 6 (6/50)...
Subject 5 processed and saved successfully!
Processing subject 6 (6/50)...
Subject 6 processed and saved successfully!
Processing subject 7 (7/50)...
Subject 6 processed and saved successfully!
Processing subject 7 (7/50)...
Subject 7 processed and saved successfully!
Processing subject 8 (8/5

Percentile (0,1) or z_score NORMALIZATION


In [16]:
# Data normalization and preparation for deep learning

def normalize_volume(volume, method='min_max'):
    """
    Normalize a 3D volume for neural network training
    
    Args:
        volume: 3D numpy array
        method: Normalization method ('min_max', 'z_score', or 'percentile')
        
    Returns:
        Normalized volume
    """
    if method == 'min_max':
        # Min-max scaling to [0,1]
        min_val = volume.min()
        max_val = volume.max()
        if max_val > min_val:
            return (volume - min_val) / (max_val - min_val)
        return volume - min_val  # Handle constant volumes
        
    elif method == 'z_score':
        # Z-score normalization (mean=0, std=1)
        mean_val = volume.mean()
        std_val = volume.std()
        if std_val > 0:
            return (volume - mean_val) / std_val
        return volume - mean_val  # Handle constant volumes
        
    elif method == 'percentile':
        # Percentile-based normalization (robust to outliers)
        p1 = np.percentile(volume, 1)
        p99 = np.percentile(volume, 99)
        if p99 > p1:
            vol_norm = (volume - p1) / (p99 - p1)
            # Clip values outside the percentile range
            return np.clip(vol_norm, 0, 1)
        return (volume - p1)  # Handle constant volumes
        
    else:
        raise ValueError(f"Unknown normalization method: {method}")

# Example usage
if 'ulf_processed' in locals() and 'hf_processed' in locals():
    # Normalize the processed volumes
    ulf_normalized = normalize_volume(ulf_processed, method='z_score')
    hf_normalized = normalize_volume(hf_processed, method='z_score')

    print("Normalization complete!")
    print(f"ULF normalized range: [{ulf_normalized.min():.4f}, {ulf_normalized.max():.4f}]")
    print(f"HF normalized range: [{hf_normalized.min():.4f}, {hf_normalized.max():.4f}]")
    
explore_3D_array_comparison(ulf_unprocessed, ulf_normalized, cmap='gray')
explore_3D_array_comparison(hf_unprocessed, hf_normalized, cmap='gray')

Normalization complete!
ULF normalized range: [-1.4882, 3.8893]
HF normalized range: [-0.7626, 3.8701]


interactive(children=(IntSlider(value=79, description='SLICE', max=159), Output()), _dom_classes=('widget-inte…

interactive(children=(IntSlider(value=79, description='SLICE', max=159), Output()), _dom_classes=('widget-inte…